# AskCampus: An NLP-Based Student Information Chatbot

**Section:** BSCS231A · **Domain:** Student information for Computer Science, Information Technology and Information System students

AskCampus answers common student questions (fees, admission, documents, offices, campus facilities). It cleans each question with NLTK, turns it into TF-IDF features, and uses a calibrated LinearSVC to decide which of 38 intents it belongs to. The reply comes from `intents.json`.

## 1. Setup

In [1]:
# Install anything that's missing. Colab already has all of these, so this
# only does something when running on a local machine.
import importlib.util
_needed = {'nltk': 'nltk', 'sklearn': 'scikit-learn', 'pandas': 'pandas', 'numpy': 'numpy',
           'matplotlib': 'matplotlib', 'seaborn': 'seaborn'}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))

import re
import string
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

# NLTK data: the tokenizer models and the WordNet dictionary used for lemmatizing.
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# A fixed seed so every run gives the same split and the same results.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Dataset

In [2]:
import json
import os

# intents.json is our only data source. It holds the example questions used for
# training and the bot's replies. (Our first draft trained on a separate list
# typed inside the notebook, which didn't match this file, so we removed it.)
# We check a few locations so the notebook runs from the GitHub repo, from Colab,
# or from the same folder as intents.json.
_candidate_paths = [
    'intents.json',
    '../data/intents.json',
    '/content/intents.json',
]
INTENTS_PATH = next((p for p in _candidate_paths if os.path.exists(p)), _candidate_paths[0])

with open(INTENTS_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []           # one row per example question: (query, intent)
response_bank = {}  # intent -> list of possible replies
skipped_entries = []

for entry in raw_data['intents']:
    intent_name = entry.get('intent')
    patterns = entry.get('text')
    responses = entry.get('responses', [])

    # Skip broken entries (no intent name or no examples) instead of crashing.
    if not intent_name or not patterns:
        skipped_entries.append(entry)
        continue

    response_bank[intent_name] = responses
    for pattern in patterns:
        rows.append({"query": pattern, "intent": intent_name})

df = pd.DataFrame(rows)

print(f"Total intents loaded: {df['intent'].nunique()}")
print(f"Total sample queries: {len(df)}")
if skipped_entries:
    print(f"Skipped {len(skipped_entries)} malformed entr"
          f"{'y' if len(skipped_entries) == 1 else 'ies'} missing 'intent' or 'text'.")

# How many examples each intent has.
df['intent'].value_counts()

Total intents loaded: 38
Total sample queries: 409


intent
course              19
admission           18
hours               17
fees                17
creator             16
document_request    16
number              15
dormitory           15
location            14
library             14
office_location     14
name                13
document            13
salutation          13
goodbye             12
vacation            12
event               11
canteen             11
sem                 11
scholarship         11
greeting            10
computerhod         10
ragging             10
placement            9
college intake       9
uniform              9
swear                9
floors               7
syllabus             7
menu                 7
principal            7
sports               7
committee            6
task                 6
facilities           5
infrastructure       3
random               3
hod                  3
Name: count, dtype: int64

In [3]:
# A random sample of the training data.
df.sample(10, random_state=RANDOM_STATE)

,query,intent
172,size of campus,floors
137,is there a dorm,dormitory
126,college address,location
94,what programs do you have,course
72,college number,number
33,who created you,creator
379,how do i get my transcript of records,document_request
223,What is average package,placement
341,give sports details,sports
227,placement,placement


In [ ]:
# Data quality check before training.
# - Duplicates matter because one copy can end up in training and the other in
#   testing, which makes the test score look better than it really is.
# - Patterns that are empty after cleaning give the model nothing to learn from.
# - Intents with very few examples are hard to learn and hard to test fairly.
_clean_preview = df['query'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.split().str.join(' ')
dupes = df[_clean_preview.duplicated(keep=False)].sort_values('query')
empty_after_cleaning = df[_clean_preview.fillna('').str.strip() == '']
counts = df['intent'].value_counts()

print(f"Duplicate patterns (same text, case/punctuation ignored): {_clean_preview.duplicated().sum()}")
print(dupes.to_string(index=False), "\n")
print(f"Patterns that are empty after removing punctuation: {len(empty_after_cleaning)}")
print(empty_after_cleaning.to_string(index=False), "\n")
print(f"Intents with fewer than 8 patterns: {(counts < 8).sum()} of {len(counts)}")
print(counts[counts < 8].to_string())

plt.figure(figsize=(12, 4))
counts.plot(kind='bar', color='#7A2E2E')
plt.title('Training patterns per intent')
plt.ylabel('Number of patterns')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.show()

## 3. Preprocessing and feature extraction

In [4]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean a question before the model sees it.

    1. Lowercase, so "Library" and "library" are the same word.
    2. Remove punctuation, so "fees?" matches "fees".
    3. Split into words with NLTK's tokenizer.
    4. Lemmatize each word to its dictionary form ("fees" -> "fee").

    We do not remove stop words. Words like "where", "who" and "when" are what
    separate many of our intents, and removing them lowered accuracy by about
    5 points in our tests (Section 7).
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(tok) for tok in tokens]
    return " ".join(lemmas)

df['clean_query'] = df['query'].apply(preprocess_text)

# Before and after cleaning.
df[['query', 'clean_query']].head(10)

,query,clean_query
0,Hi,hi
1,How are you?,how are you
2,Is anyone there?,is anyone there
3,Hello,hello
4,Good day,good day
5,What's up,whats up
6,how are ya,how are ya
7,heyy,heyy
8,whatsup,whatsup
9,??? ??? ??,


In [5]:
# Hold back 20% of the questions as a test set the model never sees in training.
# stratify keeps each intent's share the same in both parts, so even intents
# with only 3 examples appear on both sides.
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_query'], df['intent'],
    test_size=0.20, random_state=RANDOM_STATE, stratify=df['intent']
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 327
Testing samples: 82


In [6]:
# TF-IDF turns each question into numbers the model can use.
# A word scores high when it appears in the question but is rare across the
# dataset ("tuition", "registrar"), and low when it appears everywhere
# ("what", "college"). This helps the model focus on the words that tell
# intents apart.
#   ngram_range=(1, 2): use single words and pairs of words.
#   min_df=1: keep every word, since with ~400 examples a word seen once can
#             still be the only clue for a small intent.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)

# Learn the vocabulary from the training set only, then apply it to the test set.
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Training matrix shape: {X_train_tfidf.shape}")

Vocabulary size: 720
Training matrix shape: (327, 720)


## 4. Model training

In [7]:
# We train two classifiers on the same features and keep the better one.

# Baseline: Multinomial Naive Bayes, a common first choice for text.
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

# LinearSVC finds the boundary between intents with the widest margin, which
# works well when there are more features than examples (our case).
# A plain SVM gives scores, not probabilities, so we wrap it in
# CalibratedClassifierCV to get a confidence value for the fallback.
#
# The calibration step uses its own internal folds. Our first version used a
# fixed cv=3 and crashed, because some intents only have 2 examples in the
# training split. Now we pick the number of folds from the data.
min_class_count = y_train.value_counts().min()
safe_cv = max(2, min(3, min_class_count))

svc_base = LinearSVC(random_state=RANDOM_STATE, max_iter=10000)
svc_model = CalibratedClassifierCV(svc_base, cv=safe_cv)
svc_model.fit(X_train_tfidf, y_train)
svc_preds = svc_model.predict(X_test_tfidf)
svc_acc = accuracy_score(y_test, svc_preds)

print(f"Multinomial Naive Bayes accuracy: {nb_acc:.4f}")
print(f"LinearSVC (calibrated) accuracy:  {svc_acc:.4f}")

# Keep whichever model scored higher on the test set.
if svc_acc >= nb_acc:
    best_model, best_name, best_preds = svc_model, "LinearSVC", svc_preds
else:
    best_model, best_name, best_preds = nb_model, "Multinomial Naive Bayes", nb_preds

print(f"\nSelected model for chatbot deployment: {best_name}")

Multinomial Naive Bayes accuracy: 0.5976
LinearSVC (calibrated) accuracy:  0.7683

Selected model for chatbot deployment: LinearSVC


## 5. Chatbot logic

In [8]:
# If the model is less than 35% sure, the bot asks the student to rephrase
# instead of guessing. We chose 0.35 from the threshold test in Section 7.
CONFIDENCE_THRESHOLD = 0.35

fallback_response = (
    "I'm sorry, I'm not confident I understood that correctly. "
    "Could you rephrase your question, or contact the relevant office directly?"
)

def predict_intent(query, model=None, vec=None, threshold=CONFIDENCE_THRESHOLD):
    """Return (intent, confidence) for a question, or (None, confidence) if the
    model is not confident enough.

    The model and vectorizer are looked up when the function is called, not
    when it is defined. An earlier version set them as default arguments, which
    Python fixes at definition time, so the chatbot kept using the old
    evaluation model even after we trained the final one.
    """
    if model is None:
        model = best_model
    if vec is None:
        vec = vectorizer
    clean = preprocess_text(query)
    features = vec.transform([clean])

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(features)[0]
        top_idx = np.argmax(probs)
        confidence = probs[top_idx]
        intent = model.classes_[top_idx]
    else:
        intent = model.predict(features)[0]
        confidence = 1.0  # this model can't give a probability

    if confidence < threshold:
        return None, confidence
    return intent, confidence


def chatbot_response(query):
    """Pick a reply for the predicted intent, or the fallback message."""
    intent, confidence = predict_intent(query)
    if intent is None:
        return fallback_response, confidence, None
    # Replies come straight from intents.json, so there is only one place to edit them.
    candidates = response_bank.get(intent) or [fallback_response]
    return random.choice(candidates), confidence, intent

In [9]:
# Quick check with a few questions written differently from the training data.
test_queries = [
    "how much are the tuition fees this semester",
    "where is the library located",
    "can I get a copy of my documents",
    "what time does the canteen open",  # expected to be low confidence
]

for q in test_queries:
    reply, conf, intent = chatbot_response(q)
    print(f"Query: {q}")
    print(f"  -> Predicted intent: {intent} (confidence: {conf:.2f})")
    print(f"  -> Response: {reply}\n")

Query: how much are the tuition fees this semester
  -> Predicted intent: fees (confidence: 0.67)
  -> Response: Sure -- are you enrolled in Computer Science, Information Technology, or Information System? I can give you the tuition and fee breakdown for that program.

Query: where is the library located
  -> Predicted intent: library (confidence: 0.47)
  -> Response: The library is on the 2nd floor, in the middle across the hallway. It's open 8am to 6pm.

Query: can I get a copy of my documents
  -> Predicted intent: document_request (confidence: 0.47)
  -> Response: You can request official documents -- Transcript of Records, Certificate of Enrollment, Good Moral Certificate, Diploma, or Honorable Dismissal -- through the Registrar's Office. Processing usually takes 3-5 working days. Bring a valid ID and your request slip when claiming.

Query: what time does the canteen open
  -> Predicted intent: None (confidence: 0.33)
  -> Response: I'm sorry, I'm not confident I understood that 

In [ ]:
%%writefile askcampus_dialogue.py
"""
askcampus_dialogue.py

Conversation logic for AskCampus. Both the notebook chat and the Flask
backend (app.py) import this file, so they always behave the same way.

The trained classifier (TF-IDF + calibrated LinearSVC) decides the intent of
every message. This file only adds the things a one-message-at-a-time
classifier cannot handle by itself:

  * Recognizing which program a student means: full names, nicknames like
    "comsci", the short codes CS / IT / IS, and small typos.
  * Remembering that we are talking about fees, so a follow-up like "IT"
    returns the IT fee breakdown instead of being treated as a new question.
  * Answering "what are the fees for computer science?" in one step.
  * Handling a simple yes/no after the bot asks "anything else?".

Each user gets their own ConversationState, so two people chatting at the
same time never share memory.
"""

import re
import random
import string
import difflib
from dataclasses import dataclass, field
import time

import numpy as np
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

_lemmatizer = WordNetLemmatizer()

# Below this confidence the bot asks the student to rephrase instead of guessing.
CONFIDENCE_THRESHOLD = 0.35
FALLBACK_RESPONSE = (
    "I'm sorry, I'm not confident I understood that correctly. "
    "Could you rephrase your question, or contact the relevant office directly?"
)


# ---------------------------------------------------------------------------
# Text preprocessing
# This has to match the notebook's preprocess_text exactly. If the live bot
# cleaned text differently from training, it would see words the model never
# learned and accuracy would quietly drop.
# ---------------------------------------------------------------------------
def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize, and lemmatize a message."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    return " ".join(_lemmatizer.lemmatize(tok) for tok in tokens)


# ---------------------------------------------------------------------------
# Program information
# The fee amounts are sample figures for the demo, not official fees.
# ---------------------------------------------------------------------------
COURSE_FEES = {
    "computer science": {"tuition_per_unit": 1350, "laboratory": 4000, "misc": 3000, "classroom": 1000, "id_fee": 150},
    "information technology": {"tuition_per_unit": 1300, "laboratory": 3800, "misc": 3000, "classroom": 1000, "id_fee": 150},
    "information system": {"tuition_per_unit": 1250, "laboratory": 3200, "misc": 3000, "classroom": 1000, "id_fee": 150},
}

# Every way a student might name a program, mapped to its official name.
COURSE_ALIASES = {
    "computer science": "computer science", "computerscience": "computer science",
    "compsci": "computer science", "comsci": "computer science", "cs": "computer science",
    "information technology": "information technology", "infotech": "information technology",
    "info tech": "information technology", "it": "information technology",
    "information system": "information system", "information systems": "information system",
    "info system": "information system", "infosystem": "information system", "is": "information system",
}
STANDARD_LOAD_UNITS = 21

# Words we compare against when the student makes a typo (used by fuzzy matching only).
COURSE_KEYWORDS = {
    "computer science": ["computer", "science"],
    "information technology": ["information", "technology"],
    "information system": ["information", "system"],
}

# After answering one of these topics, the bot asks "anything else?".
FOLLOWUP_INTENTS = {"admission", "scholarship", "dormitory", "placement", "document", "document_request"}

# Short yes/no replies, including common Filipino ones.
YES_WORDS = {"yes", "yeah", "yep", "yup", "sure", "ok", "okay", "opo", "oo"}
NO_SINGLE_WORDS = {"no", "nope", "nah", "wala", "hindi"}
NO_PHRASES = {"no thanks", "im good", "i'm good"}


def fuzzy_match_course(message):
    """Find a program name even when it is misspelled, e.g. "compute4r science".

    Each word in the message is compared with the program keywords. Two words
    count as a match when they are at least 80% similar.
    """
    words = re.sub(r"[^a-z0-9\s]", " ", message.lower()).split()
    best_course, best_score = None, 0.0
    for course, keywords in COURSE_KEYWORDS.items():
        hits = sum(
            1 for kw in keywords
            if any(difflib.SequenceMatcher(None, w, kw).ratio() >= 0.8 for w in words)
        )
        score = hits / len(keywords)
        if score > best_score:
            best_score, best_course = score, course
    return best_course if best_score >= 0.5 else None


def match_course(message, relaxed=False):
    """Return the program named in the message, or None if there isn't one.

    The short codes "is" and "it" are also everyday English words. In an early
    version, "where is the library" was read as the IS program. So the codes
    only count when:
      - the message is already about fees (relaxed=True), or
      - the code is the whole message, like "IT", or
      - it comes at the end after a word like "about" or "for", like "what about IT".
    """
    text = message.lower().strip()
    words = text.split()

    # Full names and nicknames are safe to match anywhere in the message.
    for alias in sorted(COURSE_ALIASES, key=len, reverse=True):
        if len(alias) > 2 and re.search(rf"\b{re.escape(alias)}\b", text):
            return COURSE_ALIASES[alias]

    if relaxed:
        for code in ("cs", "it", "is"):
            if re.search(rf"\b{re.escape(code)}\b", text):
                return COURSE_ALIASES[code]
    else:
        if len(words) == 1 and words[0].strip(string.punctuation) in ("cs", "it", "is"):
            return COURSE_ALIASES[words[0].strip(string.punctuation)]
        # Matches "what about IT" or "I'm taking IS", but not "is scholarship available".
        m = re.search(r"\b(?:about|for|in|of|taking|under|sa)\s+(cs|it|is)\s*[?.!]*$", text)
        if m:
            return COURSE_ALIASES[m.group(1)]

    # Last resort: allow for typos.
    return fuzzy_match_course(message)


def build_fee_breakdown(course):
    """Build the fee reply for one program, computing the totals from COURSE_FEES."""
    fees = COURSE_FEES[course]
    tuition_total = fees["tuition_per_unit"] * STANDARD_LOAD_UNITS
    grand_total = tuition_total + fees["laboratory"] + fees["misc"] + fees["classroom"] + fees["id_fee"]
    return (
        f"Here's a generic fee breakdown for {course.title()} (based on a standard {STANDARD_LOAD_UNITS}-unit load):\n"
        f"- Tuition: PHP {fees['tuition_per_unit']:,}/unit x {STANDARD_LOAD_UNITS} units = PHP {tuition_total:,}\n"
        f"- Laboratory Fee: PHP {fees['laboratory']:,}\n"
        f"- Miscellaneous Fee: PHP {fees['misc']:,}\n"
        f"- Classroom/Facilities Fee: PHP {fees['classroom']:,}\n"
        f"- ID Fee: PHP {fees['id_fee']:,}\n"
        f"Estimated Total: PHP {grand_total:,}\n\n"
        f"Want the breakdown for another program too? Just say Computer Science, IT, or IS.\n"
        f"(These are generic sample figures for this demo -- actual fees are published each semester by the Finance Office.)"
    )


# The yes/no checks look at whole words in short replies only. An early version
# searched inside words, so "technology" was read as a "no".
def is_yes(text):
    words = re.findall(r"[a-z']+", text.lower())
    return len(words) <= 3 and any(w in YES_WORDS for w in words)


def is_no(text):
    t = text.lower().strip()
    if t in NO_PHRASES:
        return True
    words = re.findall(r"[a-z']+", t)
    return len(words) <= 3 and any(w in NO_SINGLE_WORDS for w in words)


# ---------------------------------------------------------------------------
# Conversation memory and the bot itself
# ---------------------------------------------------------------------------
@dataclass
class ConversationState:
    """What the bot remembers about one user's conversation."""
    last_topic: str = None            # e.g. "fees" while we are talking about fees
    awaiting_followup: bool = False   # True right after the bot asks "anything else?"
    last_seen: float = field(default_factory=time.time)


class AskCampusBot:
    """Holds the trained model, the vectorizer and the replies.

    The bot keeps no memory of its own. Each call to respond() gets the
    ConversationState of the user who sent the message.
    """

    def __init__(self, model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD):
        self.model = model
        self.vectorizer = vectorizer
        self.response_bank = response_bank
        self.threshold = threshold

    def classify(self, message):
        """Clean the message, turn it into TF-IDF features, and ask the model.
        Returns the top intent and its probability."""
        features = self.vectorizer.transform([preprocess_text(message)])
        probs = self.model.predict_proba(features)[0]
        top_idx = int(np.argmax(probs))
        return self.model.classes_[top_idx], float(probs[top_idx])

    def respond(self, message, state):
        """Answer one message. Returns the reply, the intent, the confidence,
        and which part of the bot answered (shown under each reply)."""
        state.last_seen = time.time()

        def out(reply, intent, confidence, handled_by):
            return {"reply": reply, "intent": intent,
                    "confidence": float(confidence), "handled_by": handled_by}

        if not message or not message.strip():
            return out(FALLBACK_RESPONSE, None, 0.0, "fallback")

        # Step 1: a yes/no answer to the bot's own "anything else?" question.
        if state.awaiting_followup:
            state.awaiting_followup = False
            if is_yes(message):
                return out("Sure! What would you like to know next -- courses, fees, "
                           "library, dormitories, admission, or scholarships?",
                           "followup_yes", 1.0, "context (follow-up)")
            if is_no(message):
                return out("Glad I could help! Feel free to ask anytime.",
                           "followup_no", 1.0, "context (follow-up)")
            # Anything else is treated as a new question.

        # Step 2: we are still talking about fees and the student named a program.
        if state.last_topic == "fees":
            course = match_course(message)
            if course:
                state.awaiting_followup = True
                return out(build_fee_breakdown(course), "fees_detail", 1.0,
                           "context (fees) + course entity")

        # Step 3: everything else goes to the machine learning model.
        intent, confidence = self.classify(message)

        if confidence < self.threshold:
            if state.last_topic == "fees":
                return out("Sorry, I didn't catch the program -- please say Computer Science, "
                           "Information Technology, or Information System.",
                           "fees_clarify", confidence, "context (fees)")
            return out(FALLBACK_RESPONSE, None, confidence, "fallback (low confidence)")

        if intent == "fees":
            state.last_topic = "fees"
            course = match_course(message, relaxed=True)
            if course:  # the program was named in the same message
                state.awaiting_followup = True
                return out(build_fee_breakdown(course), "fees_detail", confidence,
                           "model + course entity")
        else:
            state.last_topic = intent  # a new topic ends the fees conversation

        reply = random.choice(self.response_bank.get(intent) or [FALLBACK_RESPONSE])
        if intent in FOLLOWUP_INTENTS:
            reply += " Is there anything else you'd like to know?"
            state.awaiting_followup = True
        return out(reply, intent, confidence, "model")


def load_response_bank(intents_path):
    """Read the replies for each intent from intents.json, skipping broken entries."""
    import json
    with open(intents_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    bank = {}
    for entry in raw["intents"]:
        if entry.get("intent") and entry.get("text"):
            bank[entry["intent"]] = entry.get("responses", [])
    return bank

In [ ]:
# Load the dialogue module we just wrote. The reload makes sure edits are
# picked up if the cell above is run again.
import importlib
import askcampus_dialogue
importlib.reload(askcampus_dialogue)
from askcampus_dialogue import AskCampusBot, ConversationState

# Check that the module cleans text exactly like our training code did.
mismatches = [q for q in df['query'] if askcampus_dialogue.preprocess_text(q) != preprocess_text(q)]
print(f"Preprocessing parity with training pipeline: {len(df) - len(mismatches)}/{len(df)} identical")


def run_cli_chatbot(show_debug=True):
    """Chat with AskCampus in the notebook. Type 'exit' or 'quit' to stop.

    With show_debug=True, each reply also shows the intent, the confidence,
    and which part of the bot answered (the model or the dialogue layer).
    """
    # The bot is created when the chat starts, so it uses the latest model.
    bot = AskCampusBot(best_model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD)
    state = ConversationState()
    print("=" * 60)
    print(" AskCampus: Student Information Chatbot")
    print(" Type 'exit' or 'quit' to end the conversation.")
    print("=" * 60)
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print("AskCampus: Goodbye! Good luck with your enrollment concerns.")
            break
        if not user_input:
            continue
        r = bot.respond(user_input, state)
        print(f"AskCampus: {r['reply']}")
        if show_debug:
            print(f"   [intent={r['intent']}, confidence={r['confidence']:.2f}, handled by: {r['handled_by']}]")
        print()

## 6. Evaluation

In [11]:
# Scores on the 83 held-out test questions.
# Precision: when the model predicts an intent, how often is it right?
# Recall: of all questions for an intent, how many did the model find?
# F1: the balance of the two.
# "weighted" averages by how many test questions each intent has.
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, best_preds, average='weighted', zero_division=0
)

print(f"Model evaluated: {best_name}")
print(f"Accuracy:  {accuracy_score(y_test, best_preds):.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nFull classification report:\n")
print(classification_report(y_test, best_preds, zero_division=0))

Model evaluated: LinearSVC
Accuracy:  0.7683
Precision: 0.7974
Recall:    0.7683
F1-Score:  0.7547

Full classification report:

                  precision    recall  f1-score   support

       admission       1.00      0.75      0.86         4
         canteen       1.00      0.50      0.67         2
  college intake       1.00      0.50      0.67         2
       committee       1.00      1.00      1.00         1
     computerhod       1.00      0.50      0.67         2
          course       1.00      0.75      0.86         4
         creator       1.00      1.00      1.00         3
        document       0.75      1.00      0.86         3
document_request       1.00      0.67      0.80         3
       dormitory       0.75      1.00      0.86         3
           event       1.00      0.50      0.67         2
      facilities       0.50      1.00      0.67         1
            fees       1.00      1.00      1.00         3
          floors       1.00      1.00      1.00         1


In [12]:
# One 80/20 split is not very reliable for us: 11 intents have only one test
# question, so a single mistake moves that intent from 100% to 0%.
# Cross-validation splits the data into parts and tests every question once,
# which gives a fairer accuracy number.
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# The vectorizer sits inside a Pipeline so it is re-fitted in every fold and
# never sees the words of the questions it is tested on.
cv_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LinearSVC(random_state=RANDOM_STATE, max_iter=10000)),
])

# Every fold needs at least one example of each intent. Our smallest intents
# have 3 examples, so this works out to 3 folds.
min_class_count_full = df["intent"].value_counts().min()
n_splits = max(2, min(5, min_class_count_full))

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(cv_pipeline, df["clean_query"], df["intent"], cv=skf, scoring="accuracy")

print(f"{n_splits}-fold cross-validation accuracy per fold: {np.round(cv_scores, 3)}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"\nFor comparison, the single 80/20 split above gave: {accuracy_score(y_test, best_preds):.4f}")

3-fold cross-validation accuracy per fold: [0.788 0.831 0.824]
Mean CV accuracy: 0.8142 (+/- 0.0186)

For comparison, the single 80/20 split above gave: 0.7683


In [13]:
# Confusion matrix: rows are the true intent, columns are what the model
# predicted. Correct answers sit on the diagonal. The figure is also saved to
# results/ for the report and slides.
import os
labels = sorted(df['intent'].unique())
cm = confusion_matrix(y_test, best_preds, labels=labels)

plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Number of test queries'})
plt.title(f'Confusion Matrix — {best_name} (80/20 test split, {len(y_test)} queries)')
plt.xlabel('Predicted Intent')
plt.ylabel('True Intent')
plt.xticks(rotation=60, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

RESULTS_DIR = '../results' if os.path.isdir('../results') else 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR}/confusion_matrix.png")

In [14]:
# Error analysis: every test question the model got wrong.
# Grouping these, we found three causes:
#   1. Unknown words: slang like "gtg" or "okie" that never appears in training.
#   2. Overlapping intents: e.g. "sem" vs "vacation", "document" vs "document_request".
#   3. Phrasing over meaning: "do you have a name" matched "do you have a dormitory".
results_df = pd.DataFrame({
    'query': X_test.values,
    'true_intent': y_test.values,
    'predicted_intent': best_preds
})

misclassified = results_df[results_df['true_intent'] != results_df['predicted_intent']]

print(f"Number of misclassified queries: {len(misclassified)} out of {len(results_df)}")
misclassified

Number of misclassified queries: 19 out of 82


,query,true_intent,predicted_intent
0,whats there on menu,menu,greeting
9,how many stundent are taken in each branch,college intake,library
10,information system,course,sports
11,do you have a name,name,dormitory
21,cya,goodbye,swear
25,good day,greeting,salutation
30,enrolling,admission,swear
31,whatsup,greeting,swear
32,when are you guy open,hours,name
40,why i should use you,task,hours


In [ ]:
# Checking cause 1. If none of a question's words are in the vocabulary, TF-IDF
# gives it an all-zero vector, and the model falls back to the same default
# guess for every such question. That guess happens to be "swear", which is why
# slang like "cya" shows up as swear in the confusion matrix. Its confidence is
# low, so the live bot shows the "please rephrase" message instead.
# (The WordNet lemmatizer treats words as nouns, so "enrolling" never becomes
# "enroll" and ends up as an unknown word too.)
zero_feature = [q for q in X_test if vectorizer.transform([q]).nnz == 0]
print(f"Test queries with no known word or bigram: {len(zero_feature)} of {len(X_test)}")
print(zero_feature)

probe = best_model.predict_proba(vectorizer.transform(["qwertyuiop"]))[0]
print(f"\nPrediction for a nonsense word: {best_model.classes_[probe.argmax()]} (confidence {probe.max():.2f})")
print("-> every unknown input gets this same default guess, below the 0.35 threshold.")

## 7. Design experiments

In [ ]:
# In this section we test each design choice against alternatives: the
# preprocessing steps, the feature type, the classifier, and the threshold.
#
# Each option is scored with 3-fold cross-validation repeated 10 times with
# different shuffles (30 rounds), and we report the average. With only ~400
# questions, a single split is too noisy to compare options that differ by
# 1 or 2 points.
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from nltk.stem import PorterStemmer

rcv = RepeatedStratifiedKFold(n_splits=3, n_repeats=10, random_state=RANDOM_STATE)

def evaluate(name, pipeline, X, y=None):
    """Average accuracy and macro-F1 over the 30 rounds.
    Macro-F1 weighs every intent equally, so small intents count as much as big ones."""
    y = df['intent'] if y is None else y
    r = cross_validate(pipeline, X, y, cv=rcv, scoring=['accuracy', 'f1_macro'])
    return {'Option': name,
            'Accuracy': f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
            'Macro-F1': f"{r['test_f1_macro'].mean():.3f}",
            '_acc': r['test_accuracy'].mean()}

def svc_pipeline(vec=None):
    """Our usual setup: TF-IDF (or another vectorizer) followed by LinearSVC."""
    return Pipeline([('tfidf', vec or TfidfVectorizer(ngram_range=(1, 2))),
                     ('clf', LinearSVC(random_state=RANDOM_STATE, max_iter=10000))])

def show(rows):
    """Results table with each option's gap to the best one."""
    t = pd.DataFrame(rows)
    best = t['_acc'].max()
    t['vs. best'] = (t['_acc'] - best).map(lambda d: 'best' if d == 0 else f"{d*100:+.1f} pts")
    return t.drop(columns='_acc')

In [ ]:
# Experiment 1: preprocessing. The features (TF-IDF) and model (LinearSVC)
# stay the same; only the text cleaning changes.
#
# What we found: lemmatizing adds about 2 points over plain lowercasing.
# Removing stop words costs about 5 points and turns some questions, like
# "who are you", into empty strings, so we keep stop words. Stemming and
# lemmatizing verbs too score about 1 point higher, which is within the noise.
porter = PorterStemmer()

def basic_clean(t):
    t = t.lower().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', t).strip()

def remove_stopwords(t):
    return ' '.join(w for w in preprocess_text(t).split() if w not in ENGLISH_STOP_WORDS)

def porter_stem(t):
    return ' '.join(porter.stem(w) for w in word_tokenize(basic_clean(t)))

def lemmatize_verbs_too(t):
    return ' '.join(lemmatizer.lemmatize(lemmatizer.lemmatize(w, 'v'), 'n') for w in word_tokenize(basic_clean(t)))

prep_options = {
    'Lowercase only': lambda t: t.lower(),
    '+ remove punctuation, tokenize': lambda t: ' '.join(word_tokenize(basic_clean(t))),
    '+ lemmatize (OUR PIPELINE)': preprocess_text,
    '+ lemmatize + remove stop words': remove_stopwords,
    'Porter stemming instead of lemmatizing': porter_stem,
    'Lemmatize verbs too (POS-aware)': lemmatize_verbs_too,
}
prep_rows = [evaluate(name, svc_pipeline(), df['query'].apply(fn)) for name, fn in prep_options.items()]
emptied = (df['query'].apply(remove_stopwords) == '').sum()
print(f"Stop-word removal turns {emptied} of {len(df)} queries into EMPTY strings, e.g.:")
print(df.loc[df['query'].apply(remove_stopwords) == '', 'query'].head(8).tolist())
show(prep_rows)

In [ ]:
# Experiment 2: how we turn text into numbers. Preprocessing and model stay
# the same.
#
# What we found: TF-IDF beats plain word counts. Adding word pairs (bigrams)
# did not really help on our small dataset, because most pairs appear only once.
# Character n-grams (2-4 letter pieces of words) scored about 4 points higher,
# since they still partly match slang, typos and word forms like "enrolling".
# That makes them our main recommended improvement.
X_clean = df['clean_query']
feature_options = {
    'Bag of words (raw counts), unigrams': CountVectorizer(),
    'TF-IDF, unigrams': TfidfVectorizer(),
    'TF-IDF, unigrams + bigrams (OUR PIPELINE)': TfidfVectorizer(ngram_range=(1, 2)),
    'TF-IDF, 1-3 grams': TfidfVectorizer(ngram_range=(1, 3)),
    'TF-IDF, character 2-4 grams': TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)),
    'TF-IDF, words 1-2 + characters 2-4': FeatureUnion([
        ('words', TfidfVectorizer(ngram_range=(1, 2))),
        ('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)))]),
}
show([evaluate(name, svc_pipeline(vec), X_clean) for name, vec in feature_options.items()])

In [ ]:
# Experiment 3: the classifier. Preprocessing and TF-IDF features stay the same.
#
# What we found: LinearSVC is the best of the five. Naive Bayes is far behind
# because it treats words as independent and struggles with 38 intents that
# have only 3 to 20 examples each. 1-nearest-neighbour just copies the label of
# the most similar training question, so one odd example can mislead it.
model_options = {
    'Multinomial Naive Bayes (default)': MultinomialNB(),
    'Multinomial Naive Bayes (alpha=0.1)': MultinomialNB(alpha=0.1),
    'k-Nearest Neighbour (k=1, cosine)': KNeighborsClassifier(n_neighbors=1, metric='cosine'),
    'Logistic Regression': LogisticRegression(max_iter=5000, C=10),
    'LinearSVC (OUR PIPELINE)': LinearSVC(random_state=RANDOM_STATE, max_iter=10000),
}
show([evaluate(name, Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', clf)]), X_clean)
      for name, clf in model_options.items()])

In [ ]:
# Experiment 4: choosing the confidence threshold.
# Every question is scored by a model that never saw it (out-of-fold
# predictions). Then we check what would happen at different thresholds.
# A higher threshold means fewer wrong answers but more "please rephrase".
#
# What we found: at 0.35, about 93% of the answers given are correct and only
# about 5% of questions get a wrong answer, but about a third fall back. For
# student services, a wrong answer about fees is worse than asking to
# rephrase, so we accept that trade-off. The threshold looks low because the
# probability is spread across 38 intents.
from sklearn.model_selection import cross_val_predict

def calibrated(vec=None, k=2):
    """Our full model (vectorizer + calibrated LinearSVC) as one pipeline."""
    return Pipeline([('tfidf', vec or TfidfVectorizer(ngram_range=(1, 2))),
                     ('clf', CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, max_iter=10000), cv=k))])

y_all = df['intent'].values
oof = cross_val_predict(calibrated(), X_clean, y_all, method='predict_proba',
                        cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))
oof_pred = np.array(sorted(set(y_all)))[oof.argmax(1)]
oof_conf = oof.max(1)

rows = []
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    answered = oof_conf >= t
    rows.append({'Threshold': t,
                 'Questions answered': f"{answered.mean():.1%}",
                 'Accuracy of answers given': f"{(oof_pred[answered] == y_all[answered]).mean():.1%}",
                 'Wrong answers (of all questions)': f"{((oof_pred != y_all) & answered).mean():.1%}",
                 'Fallbacks': f"{(~answered).mean():.1%}"})
print(f"Average confidence of CORRECT predictions: {oof_conf[oof_pred == y_all].mean():.2f}")
pd.DataFrame(rows)

In [ ]:
# Experiment 5: robustness. Cross-validation only tests questions written in
# the same style as our training data, so we also wrote:
#   - 45 new questions in our own words (slang, synonyms, a bit of Taglish),
#     none of which appear in intents.json, and
#   - 20 off-topic questions the bot should NOT answer.
#
# What we found: only about 58% of the new questions are right. Most failures
# use words our data never contains, like "housing" (dorm), "gym" (sports) or
# "finals" (exams), because TF-IDF matches words, not meanings. About half of
# the off-topic questions still get answered. Adding character n-grams reduces
# wrong answers and rejects more off-topic questions, but doesn't fix synonyms.
UNSEEN = [
    ("hello po", "greeting"), ("good morning", "greeting"), ("yo whats good", "greeting"),
    ("thank you so much", "salutation"), ("ty", "salutation"), ("bye for now", "goodbye"), ("im heading out", "goodbye"),
    ("how much do i pay per sem", "fees"), ("magkano tuition", "fees"), ("how expensive is it to study here", "fees"),
    ("what degrees can i take", "course"), ("do you offer computer engineering", "course"),
    ("where exactly is the campus", "location"), ("whats the school address", "location"),
    ("is there housing for students", "dormitory"), ("can i live on campus", "dormitory"),
    ("when does the library open", "library"), ("where can i borrow books", "library"),
    ("what do i need to enroll", "admission"), ("how can i apply as a freshman", "admission"),
    ("how to apply for scholarship", "scholarship"), ("do you give financial assistance", "scholarship"),
    ("i need a copy of my tor", "document_request"), ("how to get my diploma", "document_request"),
    ("where is the registrar", "office_location"), ("where do i pay my tuition", "office_location"),
    ("when are the finals", "sem"), ("midterm exam schedule", "sem"), ("when is christmas break", "vacation"),
    ("who is the dean of computer studies", "computerhod"), ("whos in charge of the IT department", "computerhod"),
    ("what clubs and orgs are there", "committee"), ("is there a gym", "sports"), ("can i play basketball here", "sports"),
    ("where can i eat", "canteen"), ("what food do they sell", "menu"), ("what should i wear to class", "uniform"),
    ("who built this bot", "creator"), ("what can you help me with", "task"), ("whats your name bot", "name"),
    ("what time does school open", "hours"), ("whats the hotline", "number"), ("any upcoming events", "event"),
    ("do graduates get jobs", "placement"), ("how many students per section", "college intake"),
]
OUT_OF_SCOPE = [
    "what is the weather today", "who won the nba game last night", "how do i cook adobo",
    "can you help me with my math homework", "what is the capital of france", "recommend a good movie",
    "how to fix my laptop", "what time is it", "tell me a joke", "how do i reset my facebook password",
    "what is the price of bitcoin", "where can i buy shoes", "who is the president of the philippines",
    "how to lose weight", "translate hello to japanese", "what is love", "is it going to rain",
    "how old are you", "play some music", "order pizza for me",
]

def robustness(vec_factory, label):
    """Train on all data with the given features, then score the new and off-topic questions."""
    model = calibrated(vec_factory(), k=3).fit(X_clean, y_all)
    uq = [preprocess_text(q) for q, _ in UNSEEN]
    true = np.array([l for _, l in UNSEEN])
    pu = model.predict_proba(uq)
    pred, conf = model.classes_[pu.argmax(1)], pu.max(1)
    oos_conf = model.predict_proba([preprocess_text(q) for q in OUT_OF_SCOPE]).max(1)
    ok = conf >= CONFIDENCE_THRESHOLD
    summary = {'Features': label,
               'New phrasings: top-1 correct': f"{(pred == true).mean():.1%}",
               'answered correctly': f"{((pred == true) & ok).mean():.1%}",
               'answered WRONG': f"{((pred != true) & ok).mean():.1%}",
               'fell back': f"{(~ok).mean():.1%}",
               'Off-topic correctly rejected': f"{(oos_conf < CONFIDENCE_THRESHOLD).mean():.1%}"}
    details = pd.DataFrame({'query': [q for q, _ in UNSEEN], 'expected': true, 'predicted': pred, 'confidence': conf.round(2)})
    return summary, details, model

# Compare our current features with words + character n-grams.
word_only = lambda: TfidfVectorizer(ngram_range=(1, 2))
word_char = lambda: FeatureUnion([('words', TfidfVectorizer(ngram_range=(1, 2))),
                                  ('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)))])
s1, d1, m1 = robustness(word_only, 'Words 1-2 (OUR PIPELINE)')
s2, d2, m2 = robustness(word_char, 'Words 1-2 + characters 2-4')
display(pd.DataFrame([s1, s2]))

print("\nNew phrasings our pipeline got wrong:")
display(d1[d1['expected'] != d1['predicted']])

print("Off-topic questions our pipeline answered anyway (confidence >= threshold):")
oos_p = m1.predict_proba([preprocess_text(q) for q in OUT_OF_SCOPE])
display(pd.DataFrame([(q, m1.classes_[p.argmax()], round(p.max(), 2)) for q, p in zip(OUT_OF_SCOPE, oos_p)
                      if p.max() >= CONFIDENCE_THRESHOLD], columns=['query', 'answered as', 'confidence']))

## 8. Final model export

In [15]:
# The evaluated model only saw 80% of the data, because 20% was held back for
# testing. Now that the method is tested, we retrain the same pipeline on all
# 412 questions so the deployed bot learns from every example.
#
# We save the vectorizer and model with joblib. The Flask backend loads these
# exact files instead of training its own copy. (An earlier version of the
# backend trained separately, so the model we tested and the model in the web
# app were not the same.)
import joblib

final_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_full_tfidf = final_vectorizer.fit_transform(df["clean_query"])

min_class_count_full = df["intent"].value_counts().min()
final_safe_cv = max(2, min(3, min_class_count_full))

final_base_svc = LinearSVC(random_state=RANDOM_STATE, max_iter=10000)
final_model = CalibratedClassifierCV(final_base_svc, cv=final_safe_cv)
final_model.fit(X_full_tfidf, df["intent"])

joblib.dump(final_vectorizer, "vectorizer.joblib")
joblib.dump(final_model, "model.joblib")

print(f"Exported vectorizer.joblib (vocabulary size: {len(final_vectorizer.vocabulary_)})")
print(f"Exported model.joblib (classes: {len(final_model.classes_)})")
print("\nCopy both files AND askcampus_dialogue.py into backend/ in your repo -- app.py loads them directly.")

Exported vectorizer.joblib (vocabulary size: 824)
Exported model.joblib (classes: 38)

Copy both files into backend/ in your repo -- app.py loads them directly.


In [16]:
# From here on, the notebook's chatbot uses the final exported model,
# the same one the web app uses.
best_model = final_model
vectorizer = final_vectorizer

print("best_model and vectorizer now point to the final exported model.")
print("Try it: run_cli_chatbot() in the next cell uses this exact model.")

best_model and vectorizer now point to the final exported model.
Try it: run_cli_chatbot() in the next cell uses this exact model.


## 9. Demo

In [ ]:
# A scripted sample conversation. It doesn't wait for typing, so "Run all"
# finishes on its own. Each reply shows the intent, the confidence, and which
# part of the bot answered.
demo_bot = AskCampusBot(best_model, vectorizer, response_bank, threshold=CONFIDENCE_THRESHOLD)

def show(state, msg, who="Student A"):
    r = demo_bot.respond(msg, state)
    print(f"{who}: {msg}")
    print(f"AskCampus: {r['reply']}")
    print(f"   [intent={r['intent']}, confidence={r['confidence']:.2f}, handled by: {r['handled_by']}]\n")

student_a = ConversationState()
for msg in [
    "hi",
    "what courses do you offer",
    "how much are the fees",
    "computer science",               # answers the bot's question, handled by the dialogue layer
    "what about IT",
    "compute4r science",              # typo, still recognized
    "where is the library",           # new topic; "is" is not read as the IS program
    "what are the fees for information system",
    "how do i enroll",
    "no",
]:
    show(student_a, msg)

# Two students chatting at the same time. Each has their own memory.
print("=" * 60)
print("Two students at the same time, each with their own conversation state:\n")
student_b, student_c = ConversationState(), ConversationState()
show(student_b, "how much are the fees", who="Student B")
show(student_c, "how much are the fees", who="Student C")
show(student_b, "IT", who="Student B")                 # B gets IT fees
show(student_c, "where is the library", who="Student C")  # C moves on to a new topic
show(student_b, "what about computer science", who="Student B")  # B is still on fees

In [ ]:
# Live chat inside the notebook. Run this cell on its own, since it waits for
# you to type. Use show_debug=False to hide the intent and confidence line.
run_cli_chatbot(show_debug=True)